In [1]:
from primaite.agents.git.git_agent import GITAgent
from primaite.agents.git.git_policy import GITPolicy

from primaite.agents.llm.utils import get_obs_act_history_str, obs_diff
from primaite.environment import EnvironmentState

/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jonathan/projects/primaite/PrimAITE/src/primaite/agents/git/aegis/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2024-08-08 15:53:13.439324: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-08 15:53:14.208086: W tensorflow/compiler/tf2t

In [2]:
agent = GITAgent(training_config_path='../../config/_package_data/training/git.yaml', lay_down_config_path='../../config/_package_data/lay_down/lay_down_config_6_data_manipulation.yaml')

2024-08-08 15:53:15,364: Using: AgentFramework.CUSTOM, AgentIdentifier.GIT, ActionType.NODE, observation_space=NODE_LINK_TABLE, 512 episodes @ 8 steps
2024-08-08 15:53:15,413: Environment configuration loaded
2024-08-08 15:53:18,845: Welcome to the Primary-level AI Training Environment (PrimAITE) (version: 2.0.1)
INFO:primaite.agents.agent_abc:Welcome to the Primary-level AI Training Environment (PrimAITE) (version: 2.0.1)
2024-08-08 15:53:18,846: The output directory for this session is: /home/jonathan/primaite/2.0.1/sessions/2024-08-08/2024-08-08_15-53-15
INFO:primaite.agents.agent_abc:The output directory for this session is: /home/jonathan/primaite/2.0.1/sessions/2024-08-08/2024-08-08_15-53-15


<Figure size 640x480 with 0 Axes>

In [3]:
embs = agent._agent.llm.get_embeddings(prompt='Say hi')

In [4]:
import torch.nn.functional as F
logits = agent._agent.llm.generate_from_embeddings(text_embeddings=embs, max_new_tokens=30, grad=True)
logits = F.softmax(logits.logits[:, -1, :], dim=-1)

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


In [5]:
# What are we looking for? Any number or a .
import regex as re
filter = "^[\d.]+$"
test = '12.3.6'
print(re.fullmatch(filter, test))

<regex.Match object; span=(0, 6), match='12.3.6'>


In [22]:
# Filter out the tokenizers vocab to make sure we are only getting back what we would expect.
filter = "^[\d.]+$"
tokenizer_vocab = agent._agent.llm.tokenizer.get_vocab()
filtered_vocab = {}
for k, v in tokenizer_vocab.items():
    if re.fullmatch(filter, k):
        filtered_vocab[k] = v
    elif k == agent._agent.llm.tokenizer.eos_token:
        filtered_vocab[k] = agent._agent.llm.tokenizer.eos_token_id
    elif k == agent._agent.llm.tokenizer.bos_token:
        filtered_vocab[k] = agent._agent.llm.tokenizer.bos_token_id
    elif k == 'ass':
        filtered_vocab[k] = 520
    elif k == 'istant':
        filtered_vocab[k] = 9531
    elif k == '\\':
        filtered_vocab[k] = 76
    elif k == 'n':
        filtered_vocab[k] = 94

In [23]:
filtered_vocab

{'ass': 520,
 '........': 13548,
 '8': 40,
 '..': 950,
 '.': 30,
 '4': 36,
 '................': 26315,
 '.....': 30298,
 '<|im_end|>': 2,
 '5': 37,
 '0': 32,
 '3': 35,
 '....': 5592,
 '9': 41,
 '2': 34,
 'n': 94,
 'istant': 9531,
 '1': 33,
 '<|im_start|>': 1,
 '\\': 76,
 '6': 38,
 '...': 2026,
 '7': 39}

In [24]:
import torch
candidate_tokens = {}
for idx in range(logits.shape[-1]):
    if idx in filtered_vocab.values():
        candidate_tokens[idx] = logits[0][idx]
    
# you need to get into candidate tokens the logit and the index which is the token iD!!! find the cleanest way to do this :)

In [25]:
# You need to get the MAXIMUM tensor prob value and use the key of this as the token ID and the prob vlaue as the probabiltiy, using the grad_fn in the backward pass / loss calculation
candidate_tokens

{1: tensor(0.9971, device='cuda:0', grad_fn=<SelectBackward0>),
 2: tensor(0.0008, device='cuda:0', grad_fn=<SelectBackward0>),
 30: tensor(2.0833e-10, device='cuda:0', grad_fn=<SelectBackward0>),
 32: tensor(2.2976e-08, device='cuda:0', grad_fn=<SelectBackward0>),
 33: tensor(5.9596e-08, device='cuda:0', grad_fn=<SelectBackward0>),
 34: tensor(3.1900e-08, device='cuda:0', grad_fn=<SelectBackward0>),
 35: tensor(8.8582e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 36: tensor(7.8173e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 37: tensor(4.6680e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 38: tensor(2.8313e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 39: tensor(3.3360e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 40: tensor(1.0685e-08, device='cuda:0', grad_fn=<SelectBackward0>),
 41: tensor(1.0663e-09, device='cuda:0', grad_fn=<SelectBackward0>),
 76: tensor(6.9383e-10, device='cuda:0', grad_fn=<SelectBackward0>),
 94: tensor(3.0202e-08, device='cuda:0', gra

In [3]:
# Checking daddy's home
import torch
torch.cuda.is_available()

True

In [5]:
agent.learn()

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


assistant
Your code should be able to


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 1 has a total capacity of 10.75 GiB of which 15.62 MiB is free. Including non-PyTorch memory, this process has 10.73 GiB memory in use. Of the allocated memory 10.29 GiB is allocated by PyTorch, and 268.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)